# 🍽️ Weekly Themed Meal Planner MVP

A meal planning assistant that generates 7 themed dinners with consolidated shopping lists.

## Features
- 7 unique themed dinners (Monday-Sunday)
- 5 ingredients per recipe (excluding basics)
- Dietary preference support (vegan, vegetarian, keto, omnivore)
- Consolidated shopping list
- Email delivery

## 1. Installation & Setup

In [ ]:
%pip install -q langchain langchain-openai pydantic gradio python-dotenv

## 2. Configuration

Create a `.env` file beside this notebook. Keep real credentials out of the notebook and Git:

```text
OPENAI_API_KEY=your-openai-api-key
EMAIL_SENDER=you@gmail.com
EMAIL_PASSWORD=your-gmail-app-password
```

Email credentials are only needed when using Email Plan.

In [ ]:
from dotenv import load_dotenv

load_dotenv(".env")

## 3. Meal Planner Logic

In [ ]:
from meal_planner import (
    DietaryPreference,
    MealPlanGenerator,
    extract_json_object,
    format_ingredients,
    load_test_cases,
    normalize_optional_text,
    run_basic_validation,
    send_meal_plan_email,
)

generator = MealPlanGenerator()

## 4. Email Functionality

Email formatting and SMTP delivery are defined in `meal_planner.py`. Credentials are read from `.env` only when an email is sent.

## 5. Gradio Interface

In [ ]:
import time
import gradio as gr


def format_shopping_list(meal_plan: dict) -> str:
    """Format the shopping-list section for the app."""
    lines = ["## Shopping List\n"]
    lines.extend(
        f"- {item['total_quantity']} **{item['name']}**"
        for item in meal_plan["shopping_list"]
    )
    return "\n".join(lines)


def format_single_recipe(recipe: dict) -> str:
    """Format one recipe for the app."""
    ingredients = format_ingredients(recipe["ingredients"])
    return f"**{recipe['day']}** | {recipe['theme']} | *{recipe['dish_name']}*\n{ingredients}\n"


def generate_plan_streaming(dietary_pref, theme_requests, additional_prefs):
    """Generate and stream a plan, returning it as Gradio state."""
    yield "Generating your meal plan...", None
    result = generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=theme_requests,
        additional_preferences=additional_prefs,
    )
    if not result["success"]:
        yield f"Error: {result['error']}", None
        return

    meal_plan = result["meal_plan"]
    output = f"## Weekly {meal_plan['dietary_preference'].title()} Meal Plan\n\n"
    output += format_shopping_list(meal_plan) + "\n\n---\n\n## Recipes\n\n"
    yield output, meal_plan
    time.sleep(0.1)

    for recipe in meal_plan["recipes"]:
        output += format_single_recipe(recipe) + "\n"
        yield output, meal_plan
        time.sleep(0.05)

    validation = run_basic_validation(result)
    status = "All checks passed" if validation["passed"] else f"Issues: {', '.join(validation['errors'])}"
    output += f"---\n*Generated in {result['generation_time']}s | {status}*"
    yield output, meal_plan


def send_email_ui(email, meal_plan):
    """Send the currently displayed plan without generating another one."""
    if not email or "@" not in email:
        return "Enter a valid email address."
    if meal_plan is None:
        return "Generate a meal plan before emailing it."
    return send_meal_plan_email(email, meal_plan)["message"]


with gr.Blocks(title="Weekly Meal Planner") as app:
    gr.Markdown("# Weekly Meal Planner\nGenerate 7 themed dinners with a consolidated shopping list.")
    current_meal_plan = gr.State(value=None)

    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(
                choices=[preference.value for preference in DietaryPreference],
                value="omnivore",
                label="Diet",
            )
            theme_input = gr.Textbox(label="Themes (optional)", placeholder="Italian, Mexican...", lines=1)
            additional_input = gr.Textbox(
                label="Preferences (optional)", placeholder="no nuts, quick meals...", lines=1
            )
            generate_btn = gr.Button("Generate", variant="primary")

            gr.Markdown("---")
            email_input = gr.Textbox(label="Email", placeholder="you@example.com")
            send_btn = gr.Button("Email Plan")
            email_status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            output_display = gr.Markdown("Click **Generate** to start.")

    generate_btn.click(
        fn=generate_plan_streaming,
        inputs=[dietary_dropdown, theme_input, additional_input],
        outputs=[output_display, current_meal_plan],
    )
    send_btn.click(
        fn=send_email_ui,
        inputs=[email_input, current_meal_plan],
        outputs=[email_status],
    )

## 6. Launch Application

In [ ]:
app.launch(share=True)

## 7. Local Checks and Optional API Runs

Local checks do not call OpenAI. Set `RUN_API_TESTS` to `True` only when you intend to make four billable generation requests.

In [ ]:
from pathlib import Path

cases = load_test_cases(Path("test_cases.json"))
assert normalize_optional_text("  Italian  ", "random") == "Italian"
assert normalize_optional_text(None, "random") == "random"
assert extract_json_object('Plan: ```json\n{"ok": true}\n```') == {"ok": True}
assert format_ingredients([{"quantity": "2 cups", "name": "rice"}]) == "2 cups rice"
print(f"Local helper checks passed; loaded {len(cases)} API test cases.")

RUN_API_TESTS = False
if RUN_API_TESTS:
    for case in cases:
        result = generator.generate(**case)
        validation = run_basic_validation(result)
        status = "PASS" if validation["passed"] else validation["errors"]
        print(case["dietary_preference"], "->", status)